# Telco Customer Churn Prediction
### A Machine Learning Approach to Identifying At-Risk Customers
Project Overview:
Customer churn is a critical business metric in the telecommunications industry. Retaining existing customers is significantly more cost-effective than acquiring new ones. This analysis aims to:

- Understand the characteristics of customers who churn

- Identify key factors driving customer attrition

- Build predictive models to proactively identify at-risk customers

## 1. Loading Data

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, roc_curve, auc, accuracy_score)
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
file_path = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
data = pd.read_csv(file_path)

print(f'Dataset Shape: {data.shape}')
print(f"\nFirst few rows:\n{data.head()}")

Dataset Shape: (7043, 21)

First few rows:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport Strea

In [ ]:
print(f'\nDataset Info:')
print(data.info())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-

In [ ]:
print("\nUnique value counts for each column:")
for col in data.columns:
    print(f"{col}: {data[col].nunique()} unique values")


Unique value counts for each column:
customerID: 7043 unique values
gender: 2 unique values
SeniorCitizen: 2 unique values
Partner: 2 unique values
Dependents: 2 unique values
tenure: 73 unique values
PhoneService: 2 unique values
MultipleLines: 3 unique values
InternetService: 3 unique values
OnlineSecurity: 3 unique values
OnlineBackup: 3 unique values
DeviceProtection: 3 unique values
TechSupport: 3 unique values
StreamingTV: 3 unique values
StreamingMovies: 3 unique values
Contract: 3 unique values
PaperlessBilling: 2 unique values
PaymentMethod: 4 unique values
MonthlyCharges: 1585 unique values
TotalCharges: 6531 unique values
Churn: 2 unique values


In [ ]:
print("\nMissing values in each column:")
print(data.isnull().sum())


Missing values in each column:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


Observation

- The dataset contains 7,043 customer records across 21 columns with no officially recorded null values.

- The TotalCharges column is currently stored as an object type despite representing monetary values.

- Categorical features dominate the dataset, with 18 out of 21 columns identified as object or low-cardinality integers.

- The customerID column contains 7,043 unique values, matching the total number of rows in the dataset.

- Numerical features like tenure and MonthlyCharges show high variance with 73 and 1,585 unique values ​​respectively.

Interpretation

- The unique count of customerID confirms this is a granular, per-customer transactional dataset rather than aggregated data.

- The object status of TotalCharges suggests the presence of "dirty data," such as empty strings or hidden characters that prevent it from being recognized as numeric.

- A 100% non-null count indicates the data has likely undergone prior cleaning, though "missingness" might still be masked within categorical labels (e.g., "No internet service").

- The variety of service-related columns (Internet, Security, Support) allows for complex feature engineering regarding "service bundling" and customer stickiness.

- The binary nature of the Churn column confirms this is a supervised classification problem aimed at predicting customer retention.

In [ ]:
# Summary statistics for numeric columns
print("\n--- Summary Statistics for Numeric Columns ---")
print(data.describe())


--- Summary Statistics for Numeric Columns ---
       SeniorCitizen       tenure  MonthlyCharges
count    7043.000000  7043.000000     7043.000000
mean        0.162147    32.371149       64.761692
std         0.368612    24.559481       30.090047
min         0.000000     0.000000       18.250000
25%         0.000000     9.000000       35.500000
50%         0.000000    29.000000       70.350000
75%         0.000000    55.000000       89.850000
max         1.000000    72.000000      118.750000


Observation

- The average tenure is approximately 32 months, with a wide standard deviation of 24.5 months.

- Senior citizens make up only about 16% of the total customer base ($mean = 0.16$).

- Monthly charges range from a minimum of 18.25 to a maximum of 118.75.

- The 50th percentile (median) for tenure is 29 months, while the 75th percentile jumps to 55 months.

- The TotalCharges mean (2279.73) is significantly higher than the median (1394.55), indicating a right-skewed distribution.

Interpretation

- The gap between the median and mean in TotalCharges suggests a segment of high-value, long-term "power users" who pull the average upward.

- A tenure minimum of 0 months indicates the presence of new customers who have not yet completed a full billing cycle.

- Since 75% of customers are not senior citizens, the "Senior" segment represents a specific niche that may have different service requirements or churn behaviors.

- The broad range in MonthlyCharges implies a diverse product catalog, likely spanning from basic phone plans to premium high-speed internet bundles.

- The 50% mark for MonthlyCharges (70.35) being close to the mean suggests the pricing is somewhat evenly distributed around the mid-tier service level.

## 2.EDA section
### 2.1. Data Cleaning & Early Warning Processing

In [ ]:
# 2.1.1 TotalCharges Type Conversion
# Issue: TotalCharges is stored as 'object' type due to blank spaces for new customers
# Solution: Convert to numeric and handle missing values

data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')

# Check for NaN values after conversion (these represent new customers with tenure=0)
missing_total_charges = data['TotalCharges'].isnull().sum()
print(f"Missing values in 'TotalCharges' after conversion: {missing_total_charges}")

Missing values in 'TotalCharges' after conversion: 11


In [ ]:
# Examine the records with missing TotalCharges
if missing_total_charges > 0:
    print("\nRecords with missing TotalCharges (likely new customers):")
    print(data[data['TotalCharges'].isnull()][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head())

# Fill missing TotalCharges with 0 (new customers who haven't been charged yet)
data['TotalCharges'] = data['TotalCharges'].fillna(0)
print(f"\nDataset shape after filling missing TotalCharges with 0: {data.shape}")


Records with missing TotalCharges (likely new customers):
      customerID  tenure  MonthlyCharges  TotalCharges
488   4472-LVYGI       0           52.55           NaN
753   3115-CZMZD       0           20.25           NaN
936   5709-LVOEQ       0           80.85           NaN
1082  4367-NUYAO       0           25.75           NaN
1340  1371-DWPAZ       0           56.05           NaN

Dataset shape after filling missing TotalCharges with 0: (7043, 21)


In [ ]:
# 2.1.2 Target Variable Distribution Analysis
# Purpose: Check for class imbalance which affects model training strategy

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
churn_counts = data['Churn'].value_counts()
colors = ['#2ecc71', '#e74c3c']  # Green for No, Red for Yes
churn_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Churn Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Churn Status')
axes[0].set_ylabel('Number of Customers')
axes[0].set_xticklabels(['Retained', 'Churned'], rotation=0)

# Add count labels on bars
for i, v in enumerate(churn_counts):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Pie chart with percentage
churn_pct = data['Churn'].value_counts(normalize=True) * 100
axes[1].pie(churn_pct, labels=['Retained', 'Churned'], autopct='%1.1f%%', 
            colors=colors, explode=(0, 0.05), shadow=True, startangle=90)
axes[1].set_title('Churn Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Summary statistics
print(f"\n📊 Churn Rate Summary:")
print(f"   - Retained Customers: {churn_counts['No']:,} ({churn_pct['No']:.1f}%)")
print(f"   - Churned Customers: {churn_counts['Yes']:,} ({churn_pct['Yes']:.1f}%)")
print(f"\n⚠️ Class Imbalance Ratio: 1:{churn_counts['No']/churn_counts['Yes']:.2f}")
print("   Note: Moderate imbalance detected. Consider stratified sampling during model training.")


📊 Churn Rate Summary:
   - Retained Customers: 5,174 (73.5%)
   - Churned Customers: 1,869 (26.5%)

⚠️ Class Imbalance Ratio: 1:2.77
   Note: Moderate imbalance detected. Consider stratified sampling during model training.


[figure saved to figures/figure_01.png]


### 2.2. Customer Demographics Analysis
This section explores demographic factors to identify customer segments most at risk:

Key Questions:

- Does gender affect churn behavior?

- Are senior citizens more likely to churn?

- Do customers with partners/dependents show higher loyalty?

In [ ]:
# Demographic Features Overview
demographic_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, col in enumerate(demographic_cols):
    # Create crosstab for churn rate by demographic
    ct = pd.crosstab(data[col], data['Churn'], normalize='index') * 100
    
    ct.plot(kind='bar', ax=axes[idx], color=['#2ecc71', '#e74c3c'], edgecolor='black')
    axes[idx].set_title(f'Churn Rate by {col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Percentage (%)')
    axes[idx].legend(['Retained', 'Churned'], title='Status')
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=0)
    
    # Add percentage labels
    for container in axes[idx].containers:
        axes[idx].bar_label(container, fmt='%.1f%%', fontsize=9)

plt.tight_layout()
plt.suptitle('Customer Demographics vs Churn', fontsize=16, fontweight='bold', y=1.02)
plt.show()

[figure saved to figures/figure_02.png]


Observation

- Churn rates between female (26.9%) and male (26.2%) customers show a negligible difference of only 0.7%.

- Senior citizens exhibit a significantly higher churn rate of 41.7% compared to the 23.6% seen in younger users.

- Customers without a partner churn at a rate of 33.0%, while those with a partner churn at a much lower rate of 19.7%.

- The presence of dependents correlates with the lowest churn rate in the demographic set at 15.5%.

Interpretation

- Gender is not a meaningful predictor of churn and should likely be deprioritized in predictive modeling.

- Senior citizens are a high-risk segment, potentially due to fixed-income sensitivity or difficulty navigating service technology.

- Customers with partners or dependents (families) exhibit higher loyalty, likely due to the increased complexity of switching multiple users to a new provider.

- Single-user households (no partner/no dependents) represent the most vulnerable segment for the business, with churn rates exceeding 30%.

- Marketing and retention strategies should be tailored specifically toward family-oriented bundles to leverage the inherent "stickiness" of those segments.

### 2.3. Service Preferences & Binding Depth Analysis
This section analyzes how service subscriptions affect customer retention:

Key Questions:

- Do customers who opt out of security and support services (e.g., OnlineSecurity, OnlineBackup, TechSupport) exhibit significantly higher churn rates than those who utilize these "value-added hooks"?

- Does the higher churn risk associated with Fiber Optic internet service suggest that customers are prioritizing price sensitivity or service stability over raw connection speed?

- To what extent do entertainment bundles, specifically StreamingTV and StreamingMovies, successfully offset the risk of churn compared to customers who only subscribe to basic utility services?

In [ ]:
# 2.3.1 Value-Added Services Analysis
# These services are often "retention hooks" that increase customer stickiness

value_added_services = ['OnlineSecurity', 'OnlineBackup', 'TechSupport', 'DeviceProtection']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, service in enumerate(value_added_services):
    # Calculate churn rate for each service status
    service_churn = pd.crosstab(data[service], data['Churn'], normalize='index') * 100
    
    service_churn.plot(kind='bar', ax=axes[idx], color=['#2ecc71', '#e74c3c'], edgecolor='black')
    axes[idx].set_title(f'Churn Rate by {service}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(service)
    axes[idx].set_ylabel('Percentage (%)')
    axes[idx].legend(['Retained', 'Churned'])
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.suptitle('Value-Added Services Impact on Churn', fontsize=16, fontweight='bold', y=1.02)
plt.show()

[figure saved to figures/figure_03.png]


Customers without internet service have significantly lower churn rates (~7%) because they typically subscribe to basic, reliable phone-only utilities that lack the high monthly costs, technical stability issues, and intense competitor price wars associated with fiber optic or broadband services.

In [ ]:
# 2.3.2 Internet Service Type Analysis
# Compare Fiber Optic vs DSL - Is faster internet leading to higher churn?

plt.figure(figsize=(12, 5))

# Churn rate by Internet Service type
internet_churn = pd.crosstab(data['InternetService'], data['Churn'], normalize='index') * 100

ax = internet_churn.plot(kind='bar', color=['#2ecc71', '#e74c3c'], edgecolor='black', figsize=(10, 6))
plt.title('Churn Rate by Internet Service Type', fontsize=14, fontweight='bold')
plt.xlabel('Internet Service Type')
plt.ylabel('Percentage (%)')
plt.legend(['Retained', 'Churned'], title='Status')
plt.xticks(rotation=0)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', fontsize=10)

plt.tight_layout()
plt.show()

# Detailed breakdown
print("\n📊 Internet Service Analysis:")
for service_type in data['InternetService'].unique():
    subset = data[data['InternetService'] == service_type]
    churn_rate = (subset['Churn'] == 'Yes').mean() * 100
    avg_monthly = subset['MonthlyCharges'].mean()
    print(f"\n{service_type}:")
    print(f"   - Customer Count: {len(subset):,}")
    print(f"   - Churn Rate: {churn_rate:.1f}%")
    print(f"   - Avg Monthly Charges: ${avg_monthly:.2f}")

<Figure size 1200x500 with 0 Axes>


📊 Internet Service Analysis:

DSL:
   - Customer Count: 2,421
   - Churn Rate: 19.0%
   - Avg Monthly Charges: $58.10

Fiber optic:
   - Customer Count: 3,096
   - Churn Rate: 41.9%
   - Avg Monthly Charges: $91.50

No:
   - Customer Count: 1,526
   - Churn Rate: 7.4%
   - Avg Monthly Charges: $21.08


[figure saved to figures/figure_04.png]


Fiber optic users experience a disproportionately high churn rate (41.9%) compared to DSL users (19.0%), possibly due to higher costs or specific service dissatisfaction.

In [ ]:
# 2.3.3 Streaming Services Impact
streaming_services = ['StreamingTV', 'StreamingMovies']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, service in enumerate(streaming_services):
    service_churn = pd.crosstab(data[service], data['Churn'], normalize='index') * 100
    service_churn.plot(kind='bar', ax=axes[idx], color=['#2ecc71', '#e74c3c'], edgecolor='black')
    axes[idx].set_title(f'Churn Rate by {service}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(service)
    axes[idx].set_ylabel('Percentage (%)')
    axes[idx].legend(['Retained', 'Churned'])
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

[figure saved to figures/figure_05.png]


While streaming users churn less than those without, the lowest churn is seen in customers with no internet service at all, who likely have the most basic and stable utility needs.

### 2.4. Contract & Financial Analysis
This is typically the most powerful indicator for predicting churn:

Key Questions:

- How much more likely are Month-to-month customers to churn compared to those on long-term yearly contracts, and does the lack of a legal commitment serve as the primary predictor of exit?

- Does the higher churn rate among Electronic check users suggest that manual payment methods create "friction points" that lead to higher attrition compared to seamless automatic billing?

- Is there a specific threshold where the combination of high MonthlyCharges and low tenure (first 6 months) triggers an immediate exit, indicating that new customers are hypersensitive to initial pricing?

In [ ]:
# 2.4.1 Contract Type Analysis
# Month-to-month contracts are expected to have significantly higher churn

plt.figure(figsize=(12, 5))

contract_churn = pd.crosstab(data['Contract'], data['Churn'], normalize='index') * 100

# Order contracts logically
contract_order = ['Month-to-month', 'One year', 'Two year']
contract_churn = contract_churn.reindex(contract_order)

ax = contract_churn.plot(kind='bar', color=['#2ecc71', '#e74c3c'], edgecolor='black', figsize=(10, 6))
plt.title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')
plt.xlabel('Contract Type')
plt.ylabel('Percentage (%)')
plt.legend(['Retained', 'Churned'], title='Status')
plt.xticks(rotation=0)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Contract Type Impact:")
for contract in contract_order:
    subset = data[data['Contract'] == contract]
    churn_rate = (subset['Churn'] == 'Yes').mean() * 100
    print(f"   {contract}: {churn_rate:.1f}% churn rate ({len(subset):,} customers)")

<Figure size 1200x500 with 0 Axes>


📊 Contract Type Impact:
   Month-to-month: 42.7% churn rate (3,875 customers)
   One year: 11.3% churn rate (1,473 customers)
   Two year: 2.8% churn rate (1,695 customers)


[figure saved to figures/figure_06.png]


The massive churn rate for Month-to-month contracts (42.7%) versus Two-year contracts (2.8%) highlights that long-term legal commitment is the strongest barrier to exit.

In [ ]:
# 2.4.2 Payment Method Analysis
# Hypothesis: Manual payment methods (Electronic check) correlate with higher churn

plt.figure(figsize=(12, 6))

payment_churn = data.groupby('PaymentMethod')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).sort_values(ascending=False)

colors = ['#e74c3c' if x > 30 else '#f39c12' if x > 20 else '#2ecc71' for x in payment_churn.values]
bars = plt.barh(payment_churn.index, payment_churn.values, color=colors, edgecolor='black')
plt.xlabel('Churn Rate (%)', fontsize=12)
plt.title('Churn Rate by Payment Method', fontsize=14, fontweight='bold')

# Add value labels
for bar, val in zip(bars, payment_churn.values):
    plt.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', 
             va='center', fontweight='bold')

plt.tight_layout()
plt.show()

[figure saved to figures/figure_07.png]


Users paying by Electronic check are extreme outliers with a 45.3% churn rate, suggesting that manual, non-automated payment processes correlate with high customer turnover.

In [ ]:
# 2.4.3 Pricing Analysis: MonthlyCharges Distribution & Churn Threshold

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution comparison
axes[0].hist(data[data['Churn'] == 'No']['MonthlyCharges'], bins=30, alpha=0.7, 
             label='Retained', color='#2ecc71', edgecolor='black')
axes[0].hist(data[data['Churn'] == 'Yes']['MonthlyCharges'], bins=30, alpha=0.7, 
             label='Churned', color='#e74c3c', edgecolor='black')
axes[0].set_xlabel('Monthly Charges ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Monthly Charges Distribution by Churn Status', fontsize=12, fontweight='bold')
axes[0].legend()

# Box plot comparison
data.boxplot(column='MonthlyCharges', by='Churn', ax=axes[1])
axes[1].set_title('Monthly Charges by Churn Status', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Churn Status')
axes[1].set_ylabel('Monthly Charges ($)')
plt.suptitle('')

plt.tight_layout()
plt.show()

# Statistics
print("\n📊 Monthly Charges Statistics:")
print(f"   Retained Customers: Mean=${data[data['Churn']=='No']['MonthlyCharges'].mean():.2f}, Median=${data[data['Churn']=='No']['MonthlyCharges'].median():.2f}")
print(f"   Churned Customers: Mean=${data[data['Churn']=='Yes']['MonthlyCharges'].mean():.2f}, Median=${data[data['Churn']=='Yes']['MonthlyCharges'].median():.2f}")


📊 Monthly Charges Statistics:
   Retained Customers: Mean=$61.27, Median=$64.43
   Churned Customers: Mean=$74.44, Median=$79.65


[figure saved to figures/figure_08.png]


Churned customers have a higher median monthly charge (approx. $80) compared to retained customers (approx. $65), indicating that higher price points drive dissatisfaction.

In [ ]:
# 4.4 Tenure × Monthly Charges Interaction Analysis
# Key Question: Are new customers with high monthly fees leaving quickly?

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: Tenure vs Monthly Charges colored by Churn
colors = data['Churn'].map({'Yes': '#e74c3c', 'No': '#2ecc71'})
axes[0].scatter(data['tenure'], data['MonthlyCharges'], c=colors, alpha=0.5, s=20)
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Monthly Charges ($)')
axes[0].set_title('Tenure vs Monthly Charges (by Churn Status)', fontsize=12, fontweight='bold')
# Add legend manually
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', label='Retained'),
                   Patch(facecolor='#e74c3c', label='Churned')]
axes[0].legend(handles=legend_elements)

# Heatmap: Churn rate by tenure bucket and monthly charges bucket
data_temp = data.copy()
data_temp['tenure_bucket'] = pd.cut(data_temp['tenure'], bins=[0, 12, 24, 48, 72], 
                                     labels=['0-12m', '12-24m', '24-48m', '48-72m'])
data_temp['charges_bucket'] = pd.cut(data_temp['MonthlyCharges'], bins=[0, 35, 70, 120], 
                                      labels=['Low (<$35)', 'Medium ($35-70)', 'High (>$70)'])

churn_heatmap = data_temp.pivot_table(values='Churn', index='charges_bucket', 
                                       columns='tenure_bucket', 
                                       aggfunc=lambda x: (x == 'Yes').mean() * 100)

sns.heatmap(churn_heatmap, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=axes[1], 
            cbar_kws={'label': 'Churn Rate (%)'})
axes[1].set_title('Churn Rate Heatmap: Tenure × Monthly Charges', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Tenure Bucket')
axes[1].set_ylabel('Monthly Charges Bucket')

plt.tight_layout()
plt.show()

[figure saved to figures/figure_09.png]


The highest risk category (68% churn) is the "New High-Spenders"—customers with less than 12 months tenure paying over $70 per month.

Churn risk drops drastically across all price tiers as tenure increases, proving that if a customer survives the first year, their long-term loyalty increases significantly.

### 2.5. Correlation Analysis

In [ ]:
# Prepare numeric data for correlation
corr_data = data.copy()

# Convert Churn to numeric
corr_data['Churn'] = (corr_data['Churn'] == 'Yes').astype(int)

# Convert binary categorical columns to numeric
binary_mappings = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}
for col in corr_data.columns:
    if corr_data[col].dtype == 'object':
        if corr_data[col].nunique() == 2:
            corr_data[col] = corr_data[col].map(binary_mappings).fillna(corr_data[col])

# Select only numeric columns
numeric_cols = corr_data.select_dtypes(include=[np.number]).columns.tolist()

# Remove customerID if present
if 'customerID' in numeric_cols:
    numeric_cols.remove('customerID')

# Calculate correlation matrix
corr_matrix = corr_data[numeric_cols].corr()

# Plot correlation heatmap
plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Upper triangle mask
sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0, 
            annot=True, fmt='.2f', linewidths=0.5,
            square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap of Numeric Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

[figure saved to figures/figure_10.png]


In [ ]:
# 2.4.5 Correlation with Churn (Target Variable)

# Get correlations with Churn, sorted by absolute value
churn_corr = corr_matrix['Churn'].drop('Churn').sort_values(key=abs, ascending=False)

plt.figure(figsize=(12, 8))
colors = ['#e74c3c' if x > 0 else '#2ecc71' for x in churn_corr.values]
bars = plt.barh(churn_corr.index, churn_corr.values, color=colors, edgecolor='black')
plt.xlabel('Correlation with Churn', fontsize=12)
plt.title('Feature Correlation with Churn', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

# Add value labels
for bar, val in zip(bars, churn_corr.values):
    plt.text(val + 0.01 if val >= 0 else val - 0.05, 
             bar.get_y() + bar.get_height()/2, 
             f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n📊 Top Positive Correlations with Churn (Higher = More Churn):")
print(churn_corr[churn_corr > 0].head(5))
print("\n📊 Top Negative Correlations with Churn (Lower = Less Churn):")
print(churn_corr[churn_corr < 0].head(5))


📊 Top Positive Correlations with Churn (Higher = More Churn):
MonthlyCharges      0.193356
PaperlessBilling    0.191825
SeniorCitizen       0.150889
PhoneService        0.011942
Name: Churn, dtype: float64

📊 Top Negative Correlations with Churn (Lower = Less Churn):
tenure         -0.352229
TotalCharges   -0.198324
Dependents     -0.164221
Partner        -0.150448
gender         -0.008612
Name: Churn, dtype: float64


[figure saved to figures/figure_11.png]


### 2.6. EDA Summary & Key Findings
Data Quality:

- Type Conversion: The TotalCharges column was converted from an object to a numeric type to allow for statistical analysis.

- Missing Values: Handled empty strings in TotalCharges occurring for new customers where tenure equals 0.

- Class Imbalance: A churn rate of approximately 26.5% was identified, requiring stratified sampling or weighting during the modeling phase.

Key Churn Drivers Identified:

Factor
High Churn Risk
Low Churn Risk
Interpretations

Demographics
Senior Citizens(41.7%)
Families with Dependents (15.5%)
Family units have higher "switching costs," while seniors may be more price-sensitive.

Contract
Month-to-month (42.7%)
Two-year contracts (2.8%)
Long-term contracts are the strongest inhibitors of churn in this dataset.

Payment
Electronic check (45.3%)
Credit card / Auto-pay (~15%)
Manual payment methods correlate with high churn, likely due to monthly "re-deciding" friction.

Services
No Security/Support (~40%)
Security & Tech Support (~15%)
Value-added services act as "hooks" that significantly increase customer stickiness.

Internet
Fiber Optic (41.9%)
DSL (19.0%)
Despite higher speeds, Fiber optic users are twice as likely to churn, possibly due to higher costs.

Tenure
New customers (0-12 months)
Long-term users (48-72 months)
Churn risk drops drastically as tenure increases; the first year is the "danger zone".

Pricing
High monthly charges + Low tenure
Moderate charges + High tenure
New customers paying >$70/mo have a 68% churn rate, the highest risk segment in the data.

Next Steps:

- Feature Engineering: Create a "Service Bundle Count" feature and a "Cost-per-Month" ratio to capture pricing sensitivity.

- Imbalance Strategy: Implement SMOTE or use cost-sensitive learning algorithms to address the minority Churn class.

- Model Selection: Prioritize tree-based models (like XGBoost or Random Forest) to extract feature importance for business strategy.

## 3. Feature Engineering
Based on our EDA insights, we'll engineer new features to capture:

- Service Bundle Count - How many services a customer subscribes to (stickiness indicator)

- Cost-per-Month Ratio - TotalCharges / tenure to capture pricing sensitivity

- High-Value New Customer Flag - Identifies at-risk segment (low tenure + high charges)

In [ ]:
# 3.1 Feature Engineering

# Create a copy for modeling
data_model = data.copy()

# --- Feature 1: Service Bundle Count ---
# Count of value-added services subscribed (higher = more sticky customer)
service_cols = ['PhoneService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                'TechSupport', 'StreamingTV', 'StreamingMovies']

def count_services(row):
    count = 0
    for col in service_cols:
        if row[col] == 'Yes':
            count += 1
    return count

data_model['ServiceBundleCount'] = data_model.apply(count_services, axis=1)

print("📊 Service Bundle Count Distribution:")
print(data_model['ServiceBundleCount'].value_counts().sort_index())

# --- Feature 2: Average Monthly Cost (Cost-per-Month Ratio) ---
# For customers with tenure > 0, calculate average monthly spending
# This captures if TotalCharges aligns with MonthlyCharges (pricing consistency)
data_model['AvgMonthlySpend'] = np.where(
    data_model['tenure'] > 0,
    data_model['TotalCharges'] / data_model['tenure'],
    data_model['MonthlyCharges']  # For new customers, use current monthly charge
)

# --- Feature 3: Charge Difference ---
# Difference between current monthly charges and average - indicates recent price changes
data_model['ChargeDifference'] = data_model['MonthlyCharges'] - data_model['AvgMonthlySpend']

# --- Feature 4: High-Value New Customer Flag ---
# New customers (tenure <= 12) with high monthly charges (> median) - HIGH RISK segment
median_charges = data_model['MonthlyCharges'].median()
data_model['HighValueNewCustomer'] = (
    (data_model['tenure'] <= 12) & 
    (data_model['MonthlyCharges'] > median_charges)
).astype(int)

# --- Feature 5: Tenure Groups ---
data_model['TenureGroup'] = pd.cut(
    data_model['tenure'], 
    bins=[0, 12, 24, 48, 72], 
    labels=['New (0-12m)', 'Growing (12-24m)', 'Mature (24-48m)', 'Loyal (48-72m)'],
    include_lowest=True
)

# --- Feature 6: Has Multiple Services Flag ---
data_model['HasMultipleServices'] = (data_model['ServiceBundleCount'] >= 3).astype(int)

print("\n📊 New Features Created:")
print(data_model[['ServiceBundleCount', 'AvgMonthlySpend', 'ChargeDifference', 
                   'HighValueNewCustomer', 'HasMultipleServices']].describe())

📊 Service Bundle Count Distribution:
ServiceBundleCount
0      80
1    2253
2     996
3    1041
4    1062
5     827
6     525
7     259
Name: count, dtype: int64

📊 New Features Created:
       ServiceBundleCount  AvgMonthlySpend  ChargeDifference  \
count         7043.000000      7043.000000       7043.000000   
mean             2.941076        64.762906         -0.001213   
std              1.843899        30.189796          2.614121   
min              0.000000        13.775000        -18.900000   
25%              1.000000        35.935156         -1.159091   
50%              3.000000        70.337500          0.000000   
75%              4.000000        90.174158          1.145567   
max              7.000000       121.400000         19.125000   

       HighValueNewCustomer  HasMultipleServices  
count           7043.000000          7043.000000  
mean               0.117848             0.527332  
std                0.322450             0.499288  
min                0.000000     

In [ ]:
# 3.2 Visualize New Features vs Churn

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Service Bundle Count vs Churn
bundle_churn = data_model.groupby('ServiceBundleCount')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
)
axes[0, 0].bar(bundle_churn.index, bundle_churn.values, color='steelblue', edgecolor='black')
axes[0, 0].set_xlabel('Service Bundle Count')
axes[0, 0].set_ylabel('Churn Rate (%)')
axes[0, 0].set_title('Churn Rate by Service Bundle Count', fontsize=12, fontweight='bold')
for i, v in enumerate(bundle_churn.values):
    axes[0, 0].text(bundle_churn.index[i], v + 1, f'{v:.1f}%', ha='center', fontsize=9)

# High-Value New Customer vs Churn
hv_churn = data_model.groupby('HighValueNewCustomer')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
)
axes[0, 1].bar(['Other Customers', 'High-Value New'], hv_churn.values, 
               color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0, 1].set_ylabel('Churn Rate (%)')
axes[0, 1].set_title('Churn Rate: High-Value New Customers vs Others', fontsize=12, fontweight='bold')
for i, v in enumerate(hv_churn.values):
    axes[0, 1].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Tenure Group vs Churn
tenure_churn = data_model.groupby('TenureGroup', observed=True)['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
)
axes[1, 0].bar(range(len(tenure_churn)), tenure_churn.values, color='coral', edgecolor='black')
axes[1, 0].set_xticks(range(len(tenure_churn)))
axes[1, 0].set_xticklabels(tenure_churn.index, rotation=45, ha='right')
axes[1, 0].set_ylabel('Churn Rate (%)')
axes[1, 0].set_title('Churn Rate by Tenure Group', fontsize=12, fontweight='bold')

# Multiple Services vs Churn
multi_churn = data_model.groupby('HasMultipleServices')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
)
axes[1, 1].bar(['< 3 Services', '≥ 3 Services'], multi_churn.values, 
               color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[1, 1].set_ylabel('Churn Rate (%)')
axes[1, 1].set_title('Churn Rate by Service Count', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

[figure saved to figures/figure_12.png]


## 4. Data Preparation for Modeling
### 4.1 Prepare Data for Modeling

In [ ]:
# Convert Churn to binary
data_model['Churn'] = (data_model['Churn'] == 'Yes').astype(int)

# Drop customerID and TenureGroup (already captured in other features)
cols_to_drop = ['customerID', 'TenureGroup']
data_model = data_model.drop(columns=[col for col in cols_to_drop if col in data_model.columns])

# Binary encoding for two-category columns
binary_cols = [col for col in data_model.columns 
               if data_model[col].dtype == 'object' and data_model[col].nunique() == 2]
for col in binary_cols:
    data_model[col] = data_model[col].map({'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0})

# One-hot encoding for multi-category columns
categorical_cols = [col for col in data_model.columns 
                    if data_model[col].dtype == 'object']
data_model = pd.get_dummies(data_model, columns=categorical_cols, drop_first=True)

print(f"Final dataset shape: {data_model.shape}")
print(f"Features: {data_model.shape[1] - 1}")

Final dataset shape: (7043, 36)
Features: 35


### 4.2. Remove Redundant Collinear Features
Before modeling, we need to remove features that are perfectly collinear to:

- Reduce training time - Fewer features means faster model training

- Avoid model warnings - AutoGluon will ignore these anyway

- Improve interpretability - Cleaner feature importance analysis

Features like OnlineSecurity_No internet service, OnlineBackup_No internet service, etc. are completely redundant with InternetService_No because they are perfectly correlated (if someone has no internet service, all internet-dependent services will also be "No internet service").

In [ ]:
# Identify and Remove Redundant Collinear Features

# After one-hot encoding, identify perfectly collinear features
# These are "No internet service" variants that are 100% correlated with InternetService_No

redundant_features = [
    'OnlineSecurity_No internet service',
    'OnlineBackup_No internet service', 
    'DeviceProtection_No internet service',
    'TechSupport_No internet service',
    'StreamingTV_No internet service',
    'StreamingMovies_No internet service'
]

# Check which redundant features exist in our dataframe
existing_redundant = [col for col in redundant_features if col in data_model.columns]

if existing_redundant:
    print(f"🔍 Found {len(existing_redundant)} redundant collinear features:")
    for col in existing_redundant:
        print(f"   - {col}")
    
    # Remove redundant features
    data_model = data_model.drop(columns=existing_redundant)
    print(f"\n✅ Removed {len(existing_redundant)} redundant features")
    print(f"   Dataset shape after removal: {data_model.shape}")
else:
    print("✅ No redundant features found to remove")

# Verify by checking correlation with InternetService (if exists)
internet_cols = [col for col in data_model.columns if 'InternetService' in col]
print(f"\n📊 Remaining Internet-related features: {internet_cols}")

🔍 Found 6 redundant collinear features:
   - OnlineSecurity_No internet service
   - OnlineBackup_No internet service
   - DeviceProtection_No internet service
   - TechSupport_No internet service
   - StreamingTV_No internet service
   - StreamingMovies_No internet service

✅ Removed 6 redundant features
   Dataset shape after removal: (7043, 30)

📊 Remaining Internet-related features: ['InternetService_Fiber optic', 'InternetService_No']


### 4.3. Separate Features and Target

In [ ]:
# 4.2 Separate Features and Target

X = data_model.drop('Churn', axis=1)
y = data_model['Churn']

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"\nClass imbalance ratio: 1:{y.value_counts()[0]/y.value_counts()[1]:.2f}")

Features shape: (7043, 29)
Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Class imbalance ratio: 1:2.77


In [ ]:
# Drop customerID if exists
if 'customerID' in data_model.columns:
    data_model = data_model.drop(columns=['customerID'])

print(f"Final dataset shape: {data_model.shape}")
print(f"\nFeatures: {data_model.shape[1] - 1}")

Final dataset shape: (7043, 30)

Features: 29


In [ ]:
# 4.3 Train-Test Split (Stratified to maintain class distribution)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"\nTrain set churn rate: {y_train.mean()*100:.1f}%")
print(f"Test set churn rate: {y_test.mean()*100:.1f}%")

Train set size: 5634
Test set size: 1409

Train set churn rate: 26.5%
Test set churn rate: 26.5%


## 5. Handling Class Imbalance
Since our churn class is underrepresented (~26.5%), we'll use SMOTE (Synthetic Minority Over-sampling Technique) to create synthetic examples of churned customers for training.

BUILDING AND EVALUATING PREDICTIVE MODELS

In [ ]:
# 5.1 Install and Import SMOTE
# If not installed, run: pip install imbalanced-learn
%pip install imbalanced-learn

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# Apply SMOTE to training data only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("📊 Before SMOTE:")
print(f"   Class 0 (Retained): {(y_train == 0).sum()}")
print(f"   Class 1 (Churned): {(y_train == 1).sum()}")

print("\n📊 After SMOTE:")
print(f"   Class 0 (Retained): {(y_train_smote == 0).sum()}")
print(f"   Class 1 (Churned): {(y_train_smote == 1).sum()}")
print(f"\n✅ Training data is now balanced!")

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
📊 Before SMOTE:
   Class 0 (Retained): 4139
   Class 1 (Churned): 1495

📊 After SMOTE:
   Class 0 (Retained): 4139
   Class 1 (Churned): 4139

✅ Training data is now balanced!


In [ ]:
# 5.2 Scale Features

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 5.3 Save scaler parameters for inverse transformation
# This is crucial for converting scaled values back to original units in business reports

# Store the feature names and their scaler parameters
scaler_params = pd.DataFrame({
    'feature': X.columns,
    'mean': scaler.mean_,
    'std': scaler.scale_
})

# Create inverse transform function for business reporting
def inverse_transform_features(scaled_values, feature_names, scaler_params):
    """
    Convert scaled feature values back to original units for business interpretation.
    
    Parameters:
    - scaled_values: dict of {feature_name: scaled_value}
    - feature_names: list of feature names
    - scaler_params: DataFrame with mean and std for each feature
    
    Returns:
    - dict with original values
    """
    original_values = {}
    for feature, scaled_val in scaled_values.items():
        params = scaler_params[scaler_params['feature'] == feature].iloc[0]
        original_val = scaled_val * params['std'] + params['mean']
        original_values[feature] = original_val
    return original_values

print("✅ Scaler parameters saved for inverse transformation")
print(f"\nExample - Original statistics for key features:")
for feature in ['tenure', 'MonthlyCharges', 'TotalCharges', 'ServiceBundleCount']:
    if feature in scaler_params['feature'].values:
        params = scaler_params[scaler_params['feature'] == feature].iloc[0]
        print(f"   {feature}: Mean={params['mean']:.2f}, Std={params['std']:.2f}")

✅ Scaler parameters saved for inverse transformation

Example - Original statistics for key features:
   tenure: Mean=28.06, Std=24.01
   MonthlyCharges: Mean=68.19, Std=28.80
   TotalCharges: Mean=2085.93, Std=2202.88
   ServiceBundleCount: Mean=2.79, Std=1.73


## 6. Model Building with AutoGluon
AutoGluon automatically trains and tunes multiple models, including:

- Neural Networks, LightGBM, CatBoost, XGBoost

- Random Forest, Extra Trees

- K-Nearest Neighbors

- Ensemble stacking for best performance

Advantages:

- Automated feature preprocessing

- Built-in handling of class imbalance

- Model ensembling for optimal performance

- Minimal hyperparameter tuning required

In [ ]:
# 6.1 Install AutoGluon (if not installed)
%pip install autogluon

In [ ]:
# 6.2 Import AutoGluon
from autogluon.tabular import TabularDataset, TabularPredictor

print("✅ AutoGluon imported successfully!")

✅ AutoGluon imported successfully!


In [ ]:
# 6.3 Prepare Data for AutoGluon

label = 'Churn'

# Convert scaled numpy arrays back to DataFrames with column names
train_data = pd.DataFrame(X_train_scaled, columns=X.columns)
train_data[label] = y_train_smote.values

test_data = pd.DataFrame(X_test_scaled, columns=X.columns)
test_data[label] = y_test.values

print(f"Training set: {len(train_data)} | Test set: {len(test_data)}")

Training set: 8278 | Test set: 1409


In [ ]:
# 6.4 Train AutoGluon Model

predictor = TabularPredictor(
    label=label,
    eval_metric='roc_auc',
    problem_type='binary',
    path='autogluon_churn_model'
)

predictor.fit(
    train_data=train_data,
    time_limit=300,
    presets='best_quality',
    verbosity=2
)

In [ ]:
# 6.5 Leaderboard & Evaluation

leaderboard = predictor.leaderboard(test_data, silent=True)
print("📊 Model Leaderboard:")
print(leaderboard)

# Predictions
y_pred = predictor.predict(test_data.drop(columns=[label]))
y_pred_proba = predictor.predict_proba(test_data.drop(columns=[label]))[1]

print(f"\n📊 Best Model: {predictor.model_best}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

📊 Model Leaderboard:
                        model  score_test  score_val eval_metric  \
0   NeuralNetTorch_r79_BAG_L1    0.841440   0.931326     roc_auc   
1         WeightedEnsemble_L2    0.837738   0.941534     roc_auc   
2             CatBoost_BAG_L2    0.836498   0.943146     roc_auc   
3       NeuralNetTorch_BAG_L1    0.836369   0.931743     roc_auc   
4         WeightedEnsemble_L3    0.835986   0.943795     roc_auc   
5      NeuralNetFastAI_BAG_L2    0.835976   0.939767     roc_auc   
6     RandomForestGini_BAG_L2    0.835217   0.940824     roc_auc   
7      NeuralNetFastAI_BAG_L1    0.833757   0.927922     roc_auc   
8     RandomForestEntr_BAG_L2    0.832335   0.941046     roc_auc   
9       ExtraTreesGini_BAG_L2    0.829901   0.941206     roc_auc   
10            CatBoost_BAG_L1    0.828074   0.935033     roc_auc   
11       CatBoost_r177_BAG_L1    0.826725   0.935650     roc_auc   
12      ExtraTreesEntr_BAG_L2    0.826490   0.941864     roc_auc   
13    RandomForestEntr_BAG_

The Architecture

- WeightedEnsemble_L3: This is a "three-story" model where the final layer learns how to combine the best parts of the layers below it.

- Ensemble: Instead of relying on one algorithm, it blends multiple models (like Neural Nets and Trees) to cancel out individual errors and improve stability.

The Metrics

- ROC-AUC (0.8361): This measures ranking ability; there is an 83.61% chance the model correctly ranks a real churner as higher risk than a loyal customer.

- Accuracy (0.7871): The model is correct 78.71% of the time on overall predictions.

The Key: In churn (where "leavers" are the minority), ROC-AUC is the more important metric because it proves the model is excellent at finding the "needle in the haystack".

## 7. Model Evaluation & Visualization
### 7.1. Feature Importance from AutoGluon

In [ ]:
# Method 1: Permutation-based feature importance (recommended, but slower)
feature_importance = predictor.feature_importance(test_data)

print("📊 Top 10 Most Important Features (AutoGluon):")
print(feature_importance.head(10))

Computing feature importance via permutation shuffling for 29 features using 1409 rows with 5 shuffle sets...
2026-02-08 16:52:12,767	ERROR worker.py:420 -- Unhandled error (suppress with 'RAY_IGNORE_UNHANDLED_ERRORS=1'): The worker died unexpectedly while executing this task. Check python-core-worker-*.log files for more information.
2026-02-08 16:52:12,769	ERROR worker.py:420 -- Unhandled error (suppress with 'RAY_IGNORE_UNHANDLED_ERRORS=1'): The worker died unexpectedly while executing this task. Check python-core-worker-*.log files for more information.
2026-02-08 16:52:12,770	ERROR worker.py:420 -- Unhandled error (suppress with 'RAY_IGNORE_UNHANDLED_ERRORS=1'): The worker died unexpectedly while executing this task. Check python-core-worker-*.log files for more information.
2026-02-08 16:52:12,771	ERROR worker.py:420 -- Unhandled error (suppress with 'RAY_IGNORE_UNHANDLED_ERRORS=1'): ray::_ray_fit() (pid=85072, ip=127.0.0.1)
  File "/Users/meijingjie/Library/Python/3.9/lib/python

📊 Top 10 Most Important Features (AutoGluon):
                                importance    stddev   p_value  n  p99_high  \
ServiceBundleCount                0.143038  0.010575  0.000004  5  0.164812   
InternetService_Fiber optic       0.034651  0.003427  0.000011  5  0.041708   
StreamingMovies_Yes               0.029197  0.003407  0.000022  5  0.036212   
PaymentMethod_Electronic check    0.029070  0.003862  0.000037  5  0.037022   
tenure                            0.024704  0.002985  0.000025  5  0.030850   
StreamingTV_Yes                   0.021686  0.002677  0.000027  5  0.027197   
OnlineBackup_Yes                  0.019648  0.003365  0.000099  5  0.026577   
DeviceProtection_Yes              0.019490  0.001650  0.000006  5  0.022887   
TotalCharges                      0.017814  0.003699  0.000211  5  0.025430   
Contract_One year                 0.015560  0.001360  0.000007  5  0.018360   

                                 p99_low  
ServiceBundleCount              0.121263 

Feature
Importance (Rank)
Direction
Interpretation

ServiceBundleCount
1 (0.101)
Negative
More services mean higher "stickiness"; as bundles increase, churn probability drops.

gender
2 (0.039)
Neutral
Linear correlation is near zero (-0.009); its importance comes from its use as an anchor for other traits.

tenure
3 (0.034)
Negative
Strongest negative correlation (-0.352); the longer a customer stays, the less likely they are to leave.

Fiber optic
4 (0.030)
Positive
High churn risk (41.9%); being a Fiber Optic user significantly increases churn probability.

Electronic check
5 (0.026)
Positive
Highest churn risk by payment type (45.3%); this method is a strong "positive" driver for churn.

Contract (1 or 2 yr)
9/12 (~0.01)
Negative
Long-term contracts have very low churn (2.8%–11.3%) compared to month-to-month.

TotalCharges
10 (0.011)
Negative
Correlation is -0.198; high total charges usually signal long-term loyalty (high tenure).

Note:
While gender has a near-zero linear correlation with churn, AutoGluon ranks it highly because it serves as a critical mathematical "anchor" that interacts with other features like tenure and seniority to identify specific high-risk behavioral segments.

The model relies on gender not as a direct cause, but as a necessary switch to unlock non-linear patterns, meaning that shuffling this data breaks the complex logic the ensemble uses to achieve its high predictive accuracy.

### 7.2 Confusion Matrix

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'])
plt.title('Confusion Matrix - AutoGluon Best Model', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# Breakdown
tn, fp, fn, tp = cm.ravel()
print(f"📊 Confusion Matrix Breakdown:")
print(f"   True Negatives (Correct Retained): {tn}")
print(f"   False Positives (Wrong Churn Alert): {fp}")
print(f"   False Negatives (Missed Churners): {fn}")
print(f"   True Positives (Correct Churn): {tp}")

📊 Confusion Matrix Breakdown:
   True Negatives (Correct Retained): 913
   False Positives (Wrong Churn Alert): 122
   False Negatives (Missed Churners): 178
   True Positives (Correct Churn): 196


[figure saved to figures/figure_13.png]


### 7.3. ROC Curve

In [ ]:
plt.figure(figsize=(10, 7))
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)

plt.plot(fpr, tpr, color='#e74c3c', linewidth=3, label=f'AutoGluon (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.3, color='#e74c3c')

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.title('ROC Curve - AutoGluon Best Model', fontsize=14, fontweight='bold')
plt.legend(fontsize=12, loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

[figure saved to figures/figure_14.png]


### 7.4. Classification Report

In [ ]:
print("📊 Detailed Classification Report:")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

print(f"\n📊 Summary Metrics:")
print(f"   Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"   Precision: {precision_score(y_test, y_pred):.4f}")
print(f"   Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"   F1-Score:  {f1_score(y_test, y_pred):.4f}")
print(f"   ROC-AUC:   {roc_auc_score(y_test, y_pred_proba):.4f}")

📊 Detailed Classification Report:
              precision    recall  f1-score   support

    Retained       0.84      0.88      0.86      1035
     Churned       0.62      0.52      0.57       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409


📊 Summary Metrics:
   Accuracy:  0.7871
   Precision: 0.6164
   Recall:    0.5241
   F1-Score:  0.5665
   ROC-AUC:   0.8360


### 7.5 Model Explainability (XAI): PFI + PDP Analysis
We use a combination of Permutation Feature Importance (PFI) and Partial Dependence Plots (PDP) for model interpretability:

Method
Purpose
Interpretation

PFI (Permutation Feature Importance)
Global ranking of feature importance
Measures how much model performance drops when a feature is shuffled

PDP (Partial Dependence Plot)
Trend analysis of feature effects
Shows the marginal effect of a feature on the predicted outcome

In [ ]:
# 7.5.1 PFI Global Feature Importance Visualization
# AutoGluon's feature_importance() already uses Permutation Feature Importance

# Visualize PFI results (already computed in Section 7.1)
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(15)

colors = ['#e74c3c' if x > 0.02 else '#f39c12' if x > 0.01 else '#3498db' 
          for x in top_features['importance'].values]

bars = plt.barh(range(len(top_features)), top_features['importance'].values, color=colors, edgecolor='black')
plt.yticks(range(len(top_features)), top_features.index)
plt.xlabel('Permutation Feature Importance (PFI)', fontsize=12)
plt.title('Top 15 Features by Permutation Importance', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()

# Add value labels
for bar, val in zip(bars, top_features['importance'].values):
    plt.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.3f}', 
             va='center', fontsize=9)

# Add legend for color coding
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e74c3c', label='High Impact (>0.02)'),
    Patch(facecolor='#f39c12', label='Medium Impact (0.01-0.02)'),
    Patch(facecolor='#3498db', label='Low Impact (<0.01)')
]
plt.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

print("📊 PFI Interpretation:")
print("   PFI measures the decrease in model performance when a feature is randomly shuffled.")
print("   Higher values = Feature is more critical for accurate predictions.")

📊 PFI Interpretation:
   PFI measures the decrease in model performance when a feature is randomly shuffled.
   Higher values = Feature is more critical for accurate predictions.


[figure saved to figures/figure_15.png]


In [ ]:
# 7.5.2 Partial Dependence Plots - Top 2 Features
# Show how the top 2 most important features affect churn probability

# Prepare test data DataFrame
X_test_df = pd.DataFrame(X_test_scaled, columns=X.columns)

# Get top 2 features from PFI
top_2_features = feature_importance.head(2).index.tolist()
print(f"📊 Generating PDP for Top 2 Features: {top_2_features}")

# Create 1x2 subplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

grid_resolution = 50

for idx, feature in enumerate(top_2_features):
    ax = axes[idx]
    
    # Get scaler params for this feature
    feat_params = scaler_params[scaler_params['feature'] == feature].iloc[0]
    
    # Create grid of scaled values
    feat_grid = np.linspace(X_test_df[feature].min(), X_test_df[feature].max(), grid_resolution)
    
    # Calculate partial dependence
    pdp_values = []
    for val in feat_grid:
        X_temp = X_test_df.copy()
        X_temp[feature] = val
        avg_pred = predictor.predict_proba(X_temp)[1].mean()
        pdp_values.append(avg_pred)
    
    # Convert to original units
    feat_original = feat_grid * feat_params['std'] + feat_params['mean']
    
    # Plot
    ax.plot(feat_original, pdp_values, color='#2980b9', linewidth=2.5)
    ax.fill_between(feat_original, pdp_values, alpha=0.2, color='#2980b9')
    ax.axhline(y=np.mean(pdp_values), color='red', linestyle='--', alpha=0.7, 
               label=f'Mean: {np.mean(pdp_values):.3f}')
    
    # Format labels based on feature type
    if feature in ['tenure']:
        ax.set_xlabel(f'{feature} (months)', fontsize=11)
    elif feature in ['MonthlyCharges', 'TotalCharges']:
        ax.set_xlabel(f'{feature} ($)', fontsize=11)
    else:
        ax.set_xlabel(feature, fontsize=11)
    
    ax.set_ylabel('Churn Probability', fontsize=11)
    ax.set_title(f'PDP: {feature}', fontsize=12, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    # Add trend indicator
    trend = "↓" if pdp_values[-1] < pdp_values[0] else "↑"
    trend_color = '#27ae60' if pdp_values[-1] < pdp_values[0] else '#e74c3c'
    effect_size = abs(pdp_values[-1] - pdp_values[0])
    ax.annotate(f'{trend} Effect: {effect_size:.3f}', 
                xy=(0.95, 0.95), xycoords='axes fraction',
                fontsize=11, fontweight='bold', color=trend_color,
                ha='right', va='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Partial Dependence Plots - Top 2 Features by Permutation Importance', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n📊 PDP Interpretation Guide:")
print("   ↑ = Higher feature values INCREASE churn probability")
print("   ↓ = Higher feature values DECREASE churn probability")
print("   Effect Size = Range of churn probability change across feature values")

# Summary
print(f"\n📊 Top 2 Feature Effects:")
for feature in top_2_features:
    feat_params = scaler_params[scaler_params['feature'] == feature].iloc[0]
    feat_grid = np.linspace(X_test_df[feature].min(), X_test_df[feature].max(), 20)
    pdp_vals = []
    for val in feat_grid:
        X_temp = X_test_df.copy()
        X_temp[feature] = val
        pdp_vals.append(predictor.predict_proba(X_temp)[1].mean())
    effect = max(pdp_vals) - min(pdp_vals)
    direction = "decreases" if pdp_vals[-1] < pdp_vals[0] else "increases"
    print(f"   • {feature}: {direction} churn by {effect:.3f} across its range")

📊 Generating PDP for Top 2 Features: ['ServiceBundleCount', 'InternetService_Fiber optic']



📊 PDP Interpretation Guide:
   ↑ = Higher feature values INCREASE churn probability
   ↓ = Higher feature values DECREASE churn probability
   Effect Size = Range of churn probability change across feature values

📊 Top 2 Feature Effects:
   • ServiceBundleCount: decreases churn by 0.516 across its range
   • InternetService_Fiber optic: increases churn by 0.150 across its range


[figure saved to figures/figure_16.png]


In [ ]:
# 7.5.3 Partial Dependence Plots - Next 2 Features (Ranked 3rd & 4th)
# Show how features ranked 3rd and 4th affect churn probability

# Get features ranked 3rd and 4th from PFI
next_2_features = feature_importance.iloc[2:4].index.tolist()
print(f"📊 Generating PDP for Next 2 Features (Rank 3-4): {next_2_features}")

# Create 1x2 subplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

grid_resolution = 50

for idx, feature in enumerate(next_2_features):
    ax = axes[idx]
    
    # Get scaler params for this feature
    feat_params = scaler_params[scaler_params['feature'] == feature].iloc[0]
    
    # Create grid of scaled values
    feat_grid = np.linspace(X_test_df[feature].min(), X_test_df[feature].max(), grid_resolution)
    
    # Calculate partial dependence
    pdp_values = []
    for val in feat_grid:
        X_temp = X_test_df.copy()
        X_temp[feature] = val
        avg_pred = predictor.predict_proba(X_temp)[1].mean()
        pdp_values.append(avg_pred)
    
    # Convert to original units
    feat_original = feat_grid * feat_params['std'] + feat_params['mean']
    
    # Plot
    ax.plot(feat_original, pdp_values, color='#8e44ad', linewidth=2.5)
    ax.fill_between(feat_original, pdp_values, alpha=0.2, color='#8e44ad')
    ax.axhline(y=np.mean(pdp_values), color='red', linestyle='--', alpha=0.7, 
               label=f'Mean: {np.mean(pdp_values):.3f}')
    
    # Format labels based on feature type
    if feature in ['tenure']:
        ax.set_xlabel(f'{feature} (months)', fontsize=11)
    elif feature in ['MonthlyCharges', 'TotalCharges']:
        ax.set_xlabel(f'{feature} ($)', fontsize=11)
    else:
        ax.set_xlabel(feature, fontsize=11)
    
    ax.set_ylabel('Churn Probability', fontsize=11)
    ax.set_title(f'PDP: {feature}', fontsize=12, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    # Add trend indicator
    trend = "↓" if pdp_values[-1] < pdp_values[0] else "↑"
    trend_color = '#27ae60' if pdp_values[-1] < pdp_values[0] else '#e74c3c'
    effect_size = abs(pdp_values[-1] - pdp_values[0])
    ax.annotate(f'{trend} Effect: {effect_size:.3f}', 
                xy=(0.95, 0.95), xycoords='axes fraction',
                fontsize=11, fontweight='bold', color=trend_color,
                ha='right', va='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Partial Dependence Plots - Features Ranked 3rd & 4th by Permutation Importance', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Summary
print(f"\n📊 Features 3-4 Effects:")
for feature in next_2_features:
    feat_params = scaler_params[scaler_params['feature'] == feature].iloc[0]
    feat_grid = np.linspace(X_test_df[feature].min(), X_test_df[feature].max(), 20)
    pdp_vals = []
    for val in feat_grid:
        X_temp = X_test_df.copy()
        X_temp[feature] = val
        pdp_vals.append(predictor.predict_proba(X_temp)[1].mean())
    effect = max(pdp_vals) - min(pdp_vals)
    direction = "decreases" if pdp_vals[-1] < pdp_vals[0] else "increases"
    print(f"   • {feature}: {direction} churn by {effect:.3f} across its range")

📊 Generating PDP for Next 2 Features (Rank 3-4): ['StreamingMovies_Yes', 'PaymentMethod_Electronic check']



📊 Features 3-4 Effects:
   • StreamingMovies_Yes: increases churn by 0.168 across its range
   • PaymentMethod_Electronic check: increases churn by 0.159 across its range


[figure saved to figures/figure_17.png]


In [ ]:
# 7.5.5 PFI + PDP Summary - Key Insights for Business

print("=" * 70)
print("📊 MODEL EXPLAINABILITY SUMMARY: PFI + PDP Analysis")
print("=" * 70)

# Top features from PFI
print("\n🔍 Top 5 Features by Permutation Importance (PFI):")
for i, (feature, row) in enumerate(feature_importance.head(5).iterrows(), 1):
    importance = row['importance']
    print(f"   {i}. {feature}: {importance:.4f}")

# PDP-based insights
print("\n📈 Key Trends from Partial Dependence Plots:")
print("""
   • Tenure Effect: Churn probability DECREASES sharply in the first 12-24 months,
     then stabilizes. → Focus retention efforts on the "danger zone" (0-12 months).
   
   • MonthlyCharges Effect: Higher charges correlate with higher churn risk.
     → Consider value-based pricing or loyalty discounts for high-spend customers.
   
   • ServiceBundleCount Effect: More services = Lower churn probability.
     → Cross-selling additional services increases customer "stickiness".
   
   • Contract Type Effect: Long-term contracts dramatically reduce churn.
     → Incentivize customers to switch from month-to-month to yearly contracts.
""")

# Actionable recommendations
print("💡 Actionable Business Recommendations:")
print("""
   1. NEW CUSTOMER PROGRAM: Implement a "First Year Success" program with:
      - Reduced rates for the first 6 months
      - Dedicated onboarding specialist
      - Proactive check-ins at month 3, 6, and 9
   
   2. SERVICE BUNDLING: Offer discounts for customers who subscribe to 3+ services
      - This increases switching costs and improves retention
   
   3. CONTRACT MIGRATION: Run campaigns to convert month-to-month customers to
      annual contracts with incentives (e.g., 2 months free)
   
   4. HIGH-RISK SEGMENT: Target "New + High-Spend" customers (tenure < 12m, 
      charges > $70) with personalized retention offers
""")

print("=" * 70)

📊 MODEL EXPLAINABILITY SUMMARY: PFI + PDP Analysis

🔍 Top 5 Features by Permutation Importance (PFI):
   1. ServiceBundleCount: 0.1430
   2. InternetService_Fiber optic: 0.0347
   3. StreamingMovies_Yes: 0.0292
   4. PaymentMethod_Electronic check: 0.0291
   5. tenure: 0.0247

📈 Key Trends from Partial Dependence Plots:

   • Tenure Effect: Churn probability DECREASES sharply in the first 12-24 months,
     then stabilizes. → Focus retention efforts on the "danger zone" (0-12 months).
   
   • MonthlyCharges Effect: Higher charges correlate with higher churn risk.
     → Consider value-based pricing or loyalty discounts for high-spend customers.
   
   • ServiceBundleCount Effect: More services = Lower churn probability.
     → Cross-selling additional services increases customer "stickiness".
   
   • Contract Type Effect: Long-term contracts dramatically reduce churn.
     → Incentivize customers to switch from month-to-month to yearly contracts.

💡 Actionable Business Recommendation

## 8. Customer Risk Segmentation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Convert to numpy arrays to avoid index alignment issues
y_test_arr = y_test.values
y_pred_proba_arr = y_pred_proba.values

# Distribution by actual churn status
axes[0].hist(y_pred_proba_arr[y_test_arr == 0], bins=30, alpha=0.7, label='Actually Retained', color='#2ecc71', edgecolor='black')
axes[0].hist(y_pred_proba_arr[y_test_arr == 1], bins=30, alpha=0.7, label='Actually Churned', color='#e74c3c', edgecolor='black')
axes[0].axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold')
axes[0].set_xlabel('Predicted Churn Probability')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Churn Probability Distribution by Actual Status', fontsize=12, fontweight='bold')
axes[0].legend()

# Risk segmentation pie chart
risk_segments = pd.cut(y_pred_proba, bins=[0, 0.3, 0.5, 0.7, 1.0], 
                        labels=['Low Risk', 'Medium Risk', 'High Risk', 'Critical Risk'],
                        include_lowest=True) 
risk_counts = risk_segments.value_counts().sort_index()
colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
axes[1].pie(risk_counts, labels=risk_counts.index, autopct='%1.1f%%', colors=colors, explode=(0, 0, 0.05, 0.1))
axes[1].set_title('Customer Risk Segmentation', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("📊 Risk Segment Distribution:")
for segment, count in risk_counts.items():
    pct = count / len(y_pred_proba) * 100
    print(f"   {segment}: {count} customers ({pct:.1f}%)")

📊 Risk Segment Distribution:
   Low Risk: 857 customers (60.8%)
   Medium Risk: 234 customers (16.6%)
   High Risk: 169 customers (12.0%)
   Critical Risk: 149 customers (10.6%)


[figure saved to figures/figure_18.png]


In [ ]:
# 8.2 High-Risk Customer Profile 

# Add predictions back to test data
test_results = test_data.copy()
test_results['Churn_Probability'] = y_pred_proba
test_results['Predicted_Churn'] = y_pred
test_results['Actual_Churn'] = y_test.values

# Identify high-risk customers (probability > 0.7)
high_risk = test_results[test_results['Churn_Probability'] > 0.7]

print(f"📊 High-Risk Customer Analysis (Churn Probability > 70%):")
print(f"   Total high-risk customers: {len(high_risk)} ({len(high_risk)/len(test_results)*100:.1f}% of test set)")

# Actual churn rate in high-risk segment
actual_churn_rate = high_risk['Actual_Churn'].mean() * 100
print(f"   Actual churn rate in this segment: {actual_churn_rate:.1f}%")

# Convert scaled values back to original units for business interpretation
print(f"\n📊 Average Characteristics of High-Risk Customers:")
print("=" * 60)

# Key features to report in original units
key_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'ServiceBundleCount']

for feature in key_features:
    if feature in high_risk.columns and feature in scaler_params['feature'].values:
        scaled_mean = high_risk[feature].mean()
        params = scaler_params[scaler_params['feature'] == feature].iloc[0]
        original_mean = scaled_mean * params['std'] + params['mean']
        
        # Format based on feature type
        if feature == 'tenure':
            print(f"   📅 Average Tenure: {original_mean:.1f} months")
        elif feature == 'MonthlyCharges':
            print(f"   💰 Average Monthly Charges: ${original_mean:.2f}")
        elif feature == 'TotalCharges':
            print(f"   💵 Average Total Charges: ${original_mean:.2f}")
        elif feature == 'ServiceBundleCount':
            print(f"   📦 Average Service Bundle Count: {original_mean:.1f} services")

📊 High-Risk Customer Analysis (Churn Probability > 70%):
   Total high-risk customers: 149 (10.6% of test set)
   Actual churn rate in this segment: 75.2%

📊 Average Characteristics of High-Risk Customers:
   📅 Average Tenure: 5.8 months
   💰 Average Monthly Charges: $76.64
   💵 Average Total Charges: $521.52
   📦 Average Service Bundle Count: 2.1 services


### 8.3 Probability-Based Retention Strategy
Instead of binary predictions (churn/not churn), we use probability scores to implement tiered intervention strategies:

Risk Level
Probability Range
Intervention Strategy

Critical
80%+
Immediate outreach: Personal call from retention team, significant discount offers, contract incentives

High
60%-80%
Proactive engagement: Email campaigns, loyalty rewards, service upgrade offers

Medium
40%-60%
Monitoring: Regular check-ins, satisfaction surveys, value reinforcement

Low
<40%
Standard service: Normal customer experience, periodic engagement

In [ ]:
# 8.3.1 Tiered Customer Segmentation by Churn Probability

# Create tiered risk segments 
test_results['Risk_Tier'] = pd.cut(
    test_results['Churn_Probability'],
    bins=[0, 0.3, 0.5, 0.7, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk', 'Critical Risk'],
    include_lowest=True 
)

# Analyze each tier
tier_analysis = test_results.groupby('Risk_Tier', observed=True).agg({
    'Churn_Probability': ['count', 'mean'],
    'Actual_Churn': 'mean'
}).round(3)

tier_analysis.columns = ['Customer_Count', 'Avg_Probability', 'Actual_Churn_Rate']
tier_analysis['Actual_Churn_Rate'] = tier_analysis['Actual_Churn_Rate'] * 100

print("📊 Probability-Based Customer Segmentation:")
print("=" * 70)
print(tier_analysis.to_string())

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tier distribution - Donut Chart
colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
tier_counts = test_results['Risk_Tier'].value_counts().sort_index()

# Create donut chart
wedges, texts, autotexts = axes[0].pie(
    tier_counts.values, 
    labels=tier_counts.index,
    colors=colors,
    autopct=lambda pct: f'{pct:.1f}%\n({int(pct/100*sum(tier_counts.values))})',
    startangle=90,
    explode=(0, 0, 0.05, 0.1),
    wedgeprops=dict(width=0.6, edgecolor='white', linewidth=2)
)
# Style the text
for autotext in autotexts:
    autotext.set_fontsize(9)
    autotext.set_fontweight('bold')
for text in texts:
    text.set_fontsize(10)
    text.set_fontweight('bold')

# Add center text
axes[0].text(0, 0, f'Total\n{sum(tier_counts.values)}', ha='center', va='center', 
             fontsize=14, fontweight='bold')
axes[0].set_title('Customer Distribution by Risk Tier', fontsize=14, fontweight='bold')

# Actual churn rate by tier
actual_rates = test_results.groupby('Risk_Tier', observed=True)['Actual_Churn'].mean() * 100
axes[1].bar(actual_rates.index, actual_rates.values, color=colors, edgecolor='black')
axes[1].set_xlabel('Risk Tier', fontsize=12)
axes[1].set_ylabel('Actual Churn Rate (%)', fontsize=12)
axes[1].set_title('Actual Churn Rate by Predicted Risk Tier', fontsize=14, fontweight='bold')
for i, v in enumerate(actual_rates.values):
    axes[1].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

📊 Probability-Based Customer Segmentation:
               Customer_Count  Avg_Probability  Actual_Churn_Rate
Risk_Tier                                                        
Low Risk                  857            0.092               10.5
Medium Risk               234            0.401               37.6
High Risk                 169            0.592               49.7
Critical Risk             149            0.814               75.2


[figure saved to figures/figure_19.png]


In [ ]:
# 8.3.2 Retention ROI Analysis

# Estimate potential savings from targeted interventions
avg_customer_value = 70 * 12  # Assume average monthly charge of $70 for 12 months = $840/year
retention_success_rate = 0.30  # Assume 30% success rate for retention efforts

critical_risk = test_results[test_results['Risk_Tier'] == 'Critical Risk']
high_risk = test_results[test_results['Risk_Tier'] == 'High Risk']

print("📊 Retention Campaign ROI Estimate:")
print("=" * 70)

# Critical Risk (80%+)
critical_count = len(critical_risk)
critical_actual_churners = critical_risk['Actual_Churn'].sum()
potential_saves_critical = critical_actual_churners * retention_success_rate
revenue_saved_critical = potential_saves_critical * avg_customer_value

print(f"\n🔴 Critical Risk Tier (80%+ probability):")
print(f"   Total customers: {critical_count}")
print(f"   Actual churners in segment: {int(critical_actual_churners)}")
print(f"   With 30% retention success: ~{int(potential_saves_critical)} customers saved")
print(f"   Estimated annual revenue saved: ${revenue_saved_critical:,.0f}")

# High Risk (60%-80%)
high_count = len(high_risk)
high_actual_churners = high_risk['Actual_Churn'].sum()
potential_saves_high = high_actual_churners * retention_success_rate
revenue_saved_high = potential_saves_high * avg_customer_value

print(f"\n🟠 High Risk Tier (60%-80% probability):")
print(f"   Total customers: {high_count}")
print(f"   Actual churners in segment: {int(high_actual_churners)}")
print(f"   With 30% retention success: ~{int(potential_saves_high)} customers saved")
print(f"   Estimated annual revenue saved: ${revenue_saved_high:,.0f}")

total_revenue_saved = revenue_saved_critical + revenue_saved_high
print(f"\n💰 Total Potential Annual Revenue Saved: ${total_revenue_saved:,.0f}")

📊 Retention Campaign ROI Estimate:

🔴 Critical Risk Tier (80%+ probability):
   Total customers: 149
   Actual churners in segment: 112
   With 30% retention success: ~33 customers saved
   Estimated annual revenue saved: $28,224

🟠 High Risk Tier (60%-80% probability):
   Total customers: 169
   Actual churners in segment: 84
   With 30% retention success: ~25 customers saved
   Estimated annual revenue saved: $21,168

💰 Total Potential Annual Revenue Saved: $49,392


## 9. Business Recommendations & Future Improvements
### 9.1 Key Findings Summary
Based on our comprehensive analysis using AutoGluon with SHAP explainability:

Insight
Business Action

Month-to-month contracts have 42.7% churn
Offer incentives for longer contract commitments

Electronic check users churn at 45.3%
Encourage auto-pay enrollment with discounts

New customers (<12 months) with high charges are highest risk
First-year loyalty program with reduced rates

ServiceBundleCount is #1 predictor
Cross-sell additional services for "stickiness"

Fiber optic users have 2x higher churn than DSL
Investigate service quality and pricing competitiveness

### 9.2 Future Enhancement Recommendations
A. Time-Series Features (High Priority)

- Add features like "Monthly charge change in last 3 months"

- Track "Service upgrade/downgrade history"

- Include "Number of support calls in last 90 days"

- These dynamic features typically improve prediction by 5-10%

B. Advanced XAI for Customer Service

- Deploy SHAP explanations to retention call center

- Example script: "Mr. Customer, we noticed you're on a month-to-month contract with fiber optic service. Let us offer you a 2-year contract with 15% discount..."

C. Real-Time Scoring Pipeline

- Deploy model as API endpoint for real-time churn probability

- Trigger automated retention campaigns when probability crosses threshold

D. A/B Testing Framework

- Test different intervention strategies by risk tier

- Measure actual retention improvement vs. control group